In [3]:
# Scenario: AI-Powered Study Assistant (Flashcard-Based)
# Imagine an educational platform has deployed a study assistant that generates
# flashcards for learners by calling the Groq API. The workflow is modeled as a
# graph of states, where each learner query flows through nodes until a
# flashcard is delivered and evaluated.

# 1. State Definition
# The study assistant maintains a notebook-like state:
# - topic → The subject the learner wants to study.
# - flashcard → The generated flashcard question.
# - correct_answer → The answer generated by Groq.
# - learner_answer → The learner’s submitted answer.
# - feedback → Whether the learner was correct or not.
# - progress → A log of all past flashcards attempted.

from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class StudyState(TypedDict):
    topic: str
    flashcard: str
    correct_answer: str
    learner_answer: str
    feedback: str
    progress: list

# 2. Define Nodes (functions)

# This node calls the Groq API to generate one flashcard
# based on the topic entered by the learner.
def generate_flashcard(state: StudyState):
    topic = state["topic"]

    # Fetch Groq API key from Colab secrets
    groq_api_key = userdata.get('groq_api_key')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'groq_api_key'.")

    # Call Groq API to generate one flashcard in fixed format
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {
                    "role": "user",
                    "content": (
                        f"Generate one simple study flashcard on the topic '{topic}' "
                        f"in this exact format only:\n"
                        f"Question: <question>\n"
                        f"Answer: <answer>"
                    )
                }
            ],
        }
    )

    # Handle API errors safely
    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()

    # Validate API response format
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    content = response_json["choices"][0]["message"]["content"]

    # Extract question and answer from the generated content
    lines = content.split("\n")
    question = ""
    answer = ""

    for line in lines:
        if line.lower().startswith("question:"):
            question = line.split(":", 1)[1].strip()
        elif line.lower().startswith("answer:"):
            answer = line.split(":", 1)[1].strip()

    if not question or not answer:
        raise ValueError(f"Could not parse flashcard properly. Full response: {content}")

    return {
        "flashcard": question,
        "correct_answer": answer
    }

# This node formats the flashcard question for display
def present_flashcard(state: StudyState):
    print("\nFlashcard Question:", state["flashcard"])
    return {"feedback": f"Flashcard Question: {state['flashcard']}"}

# This node checks whether the learner's answer matches the correct answer
def evaluate_answer(state: StudyState):
    learner_ans = state["learner_answer"].strip().lower()
    correct_ans = state["correct_answer"].strip().lower()

    if learner_ans == correct_ans:
        result = "Correct!"
    else:
        result = f"Incorrect. Correct answer: {state['correct_answer']}"

    return {"feedback": result}

# This node stores the learner's attempt in the progress log
def update_progress(state: StudyState):
    attempt = {
        "topic": state["topic"],
        "question": state["flashcard"],
        "learner_answer": state["learner_answer"],
        "correct_answer": state["correct_answer"],
        "result": state["feedback"]
    }

    return {
        "progress": state["progress"] + [attempt]
    }

# 3. Build the Graph
graph = StateGraph(StudyState)
graph.add_node("generate_flashcard", generate_flashcard)
graph.add_node("present_flashcard", present_flashcard)
graph.add_node("evaluate_answer", evaluate_answer)
graph.add_node("update_progress", update_progress)

# 4. Add Edges
graph.set_entry_point("generate_flashcard")
graph.add_edge("generate_flashcard", "present_flashcard")
graph.add_edge("present_flashcard", "evaluate_answer")
graph.add_edge("evaluate_answer", "update_progress")
graph.add_edge("update_progress", END)

# 5. Example Run
if __name__ == "__main__":
    # Take topic input from learner
    user_topic = input("Enter the topic you want to study: ")

    # Temporary initial state before learner sees the generated question
    initial_state = {
        "topic": user_topic,
        "flashcard": "",
        "correct_answer": "",
        "learner_answer": "",
        "feedback": "",
        "progress": []
    }

    # Compile the graph
    app = graph.compile()

    # Step 1: Generate flashcard first
    result = app.invoke(initial_state)

    # Step 2: Ask learner for answer after showing the flashcard question
    learner_response = input("\nEnter your answer: ")

    # Step 3: Re-run with learner answer included
    updated_state = {
        "topic": result["topic"],
        "flashcard": result["flashcard"],
        "correct_answer": result["correct_answer"],
        "learner_answer": learner_response,
        "feedback": "",
        "progress": result["progress"]
    }

    final_result = app.invoke(updated_state)

    # Final output
    print("\nFeedback:", final_result["feedback"])
    print("\nProgress Log:")
    for item in final_result["progress"]:
        print(item)

Enter the topic you want to study: science

Flashcard Question: What is the process by which plants convert sunlight into energy?

Enter your answer: photosynthesis

Flashcard Question: What is the process of converting energy from one form to another called?

Feedback: Incorrect. Correct answer: Energy transformation

Progress Log:
{'topic': 'science', 'question': 'What is the process by which plants convert sunlight into energy?', 'learner_answer': '', 'correct_answer': 'Photosynthesis', 'result': 'Incorrect. Correct answer: Photosynthesis'}
{'topic': 'science', 'question': 'What is the process of converting energy from one form to another called?', 'learner_answer': 'photosynthesis', 'correct_answer': 'Energy transformation', 'result': 'Incorrect. Correct answer: Energy transformation'}
